## <국립민속박물관-흉배 스크래핑>

### 0. 필요한 패키지 설치
- beautifulsoup, pandas, openpyxl

In [ ]:
# !pip install beautifulsoup4 pandas openpyxl requests -q

### 1. 접속 준비 (세션 만들기, CSRF 토큰 만들기)
- 국립민속박물관 검색 기능은 보안을 위해 **CSRF 토큰**이라는 임시 값을 요구
    - 이 요청이 실제로 저 검색 페이지를 열어본 사람이 보낸 게 맞다는 걸 증명하기 위함
- 먼저 검색 페이지에 평범하게 접속(GET)해서, 페이지 안에 숨어있는 CSRF 토큰 값 읽기
- 이때 받은 쿠키(로그인 상태 같은 걸 기억하는 값)와 토큰을 계속 재사용해서 이후 요청 보내기

In [1]:
# 1. 필요한 패키지, 라이브러리 불러오기
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# 공통 함수는 utils.py에 모아두고 여기서 가져다 씀 (01~03 노트북이 다 같은 함수를 공유)
from utils import BASE, get_csrf_token, search_relic_list, get_relic_detail

# 2. URL
SEARCH_LIST_URL = f"{BASE}/user/data/home/101/DataRelicCategoryList.do"
DETAIL_URL = f"{BASE}/user/data/home/101/DataRelicView.do"

# 실제 브라우저처럼 보이도록 User-Agent를 지정해줌 -> 일부 서버는 이게 없으면 차단하기 때문
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

# 3. 쿠키를 자동으로 기억해주는 과정 (쿠키 관리해올 필요 없게)
session = requests.Session()
session.headers.update(HEADERS)

# 4. CSRF 토큰 읽기 (utils.py의 함수 사용, session을 인자로 넘겨줌)
csrf_token = get_csrf_token(session)        # utils.py에 있는 함수
print("CSRF 토큰:", csrf_token)

CSRF 토큰: 9ca2d97c-5295-4eb6-9bc3-57f99351183d


### 2. 데이터 수집
- 키워드 기반 수집 -> 첫 페이지 수집 -> 상세 페이지 내용 수집

In [2]:
# 1. 키워드 기반 수집
KEYWORDS = ["흉배"]

all_list_items = []
for kw in KEYWORDS:
    all_list_items.extend(search_relic_list(session, csrf_token, kw))   # search_relic_test는 utils.py에 있음

print(len(all_list_items))

123


In [3]:
# 2. 데이터 프레임 제작
list_df = pd.DataFrame(all_list_items)
list_df = list_df.drop_duplicates(subset="seq").reset_index(drop=True)
print("중복 제거 후 소장품 수:", len(list_df))
list_df.head()  # 여기서 흉배판이 있기 때문에 흉배판은 제함

중복 제거 후 소장품 수: 123


,seq,list_title,searched_keyword
0,PS0100200100100508000000,흉배(胸背),흉배
1,PS0100200100100508100000,흉배(胸背),흉배
2,PS0100200100100078800000,흉배판(胸背板),흉배
3,PS0100200100110000400000,흉배판(胸背板),흉배
4,PS0100200100100078700000,흉배판(胸背板),흉배


In [4]:
# 3. 상세 페이지 열기
details = []
for i, row in list_df.iterrows():
    detail = get_relic_detail(session, row["seq"])      # get_relic_detail은 utils.py에 있음
    detail["searched_keyword"] = row["searched_keyword"]
    detail["list_title"] = row["list_title"]
    details.append(detail)
    if (i + 1) % 20 == 0:
        print(f"{i + 1} / {len(list_df)} 건 완료")

print("총", len(details), "건")

20 / 123 건 완료
40 / 123 건 완료
60 / 123 건 완료
80 / 123 건 완료
100 / 123 건 완료
120 / 123 건 완료
총 123 건


### 3. 표로 정리 + 파일 저장

In [9]:
# 1. 데이터프레임 확인
df = pd.DataFrame(details)

# 2. 컬럼 순서 정리
priority_cols = ["searched_keyword", "소장품 명칭", "list_title", "국적/시대", "용도/기능",
                 "크기", "소장품 번호", "내용", "image_url", "detail_url", "seq"]
other_cols = [c for c in df.columns if c not in priority_cols]
ordered_cols = [c for c in priority_cols if c in df.columns] + other_cols
df = df[ordered_cols]


pure_hyungbae = df[
    df["소장품 명칭"].str.contains("흉배", na=False) &   # "흉배"가 들어있으면서
    ~df["소장품 명칭"].str.contains("흉배판", na=False)  # "흉배판"은 아닌 것
]

print(pure_hyungbae.shape)
pure_hyungbae.head()

(23, 12)


,searched_keyword,소장품 명칭,list_title,국적/시대,용도/기능,크기,소장품 번호,내용,image_url,detail_url,seq,image_urls
0,흉배,흉배(胸背),흉배(胸背),한국-조선,의-의류-부분품-보/흉배,세로 : 26.7 가로 : 26,005080,조선시대 무관 중 당상관이 평상복의 앞뒤에 붙여 품계를 나타내던 표지. 청녹색 비단...,https://www.nfm.go.kr/common/apiimage/relic/82...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100100508000000,https://www.nfm.go.kr/common/apiimage/relic/82...
1,흉배,흉배(胸背),흉배(胸背),한국-조선,의-의류-부분품-보/흉배,세로 : 21.5 가로 : 20,005081,조선시대 무관 중 당하관이 평상복의 앞뒤에 붙여 품계를 나타내던 표지. 청녹색 비단...,https://www.nfm.go.kr/common/apiimage/relic/82...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100100508100000,https://www.nfm.go.kr/common/apiimage/relic/82...
6,흉배,흉배(胸背),흉배(胸背),한국,의-의류-부분품-보/흉배,세로 : 20.7 가로 : 20.6,029184,조선 시대에 문무관(文武官)이 입는 관복의 가슴과 등에 학이나 범을 수 놓아 붙이던...,https://www.nfm.go.kr/common/apiimage/relic/83...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100102918400000,https://www.nfm.go.kr/common/apiimage/relic/83...
7,흉배,웅비(熊비)흉배,웅비(熊비)흉배,한국-광복이후,의-의류-부분품-보/흉배,가로 : 16.2 세로 : 17,000915,무관 3품이 사용하던 흉배의 재현품. 초록색 바탕에 은사로 수를 놓음. 웅비 2마리...,https://www.nfm.go.kr/common/apiimage/relic/82...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100900091500000,https://www.nfm.go.kr/common/apiimage/relic/82...
8,흉배,노사(鷺사)흉배,노사(鷺사)흉배,한국-광복이후,의-의류-부분품-보/흉배,가로 : 30 세로 : 30.5,000914,문관 6품이 사용하던 흉배의 재현품. 초록색 바탕에 금색 계열의 실로 수를 놓음. ...,https://www.nfm.go.kr/common/apiimage/relic/84...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100900091400000,https://www.nfm.go.kr/common/apiimage/relic/84...


In [6]:
# 3. 파일 저장(엑셀)
pure_hyungbae.to_excel("../data/nfm_hyungbae.xlsx", index=False)

# 4. 이미지 저장 (한 유물에 사진이 여러 장이면 -1, -2 ... 붙여서 다 저장함)
pure_hyungbae = pure_hyungbae.reset_index(drop=True) 

for i, row in pure_hyungbae.iterrows():
    seq = row["seq"]
    image_urls = str(row["image_urls"]).split("; ") if pd.notna(row["image_urls"]) else []

    for idx, image_url in enumerate(image_urls):
        suffix = "" if len(image_urls) == 1 else f"-{idx + 1}"  # 여러 장일 때만 -1, -2 ... 붙임
        filepath = f"../image/hyungbae/{seq}{suffix}.jpg"

        resp = session.get(image_url, timeout=30)
        with open(filepath, "wb") as f:
            f.write(resp.content)

    if (i + 1) % 23 == 0:
        print(f"{i + 1} / {len(pure_hyungbae)} 완료")

    time.sleep(0.5)

23 / 23 완료
